In [6]:
from pathlib import Path
import shutil
import random

# Define base paths
labels_raw_path = Path("labels_raw")
raw_images_path = Path("raw")
images_train_path = Path("images/train")
images_val_path = Path("images/val")
labels_train_path = Path("labels/train")
labels_val_path = Path("labels/val")

# Create output directories if they don't exist
for path in [images_train_path, images_val_path, labels_train_path, labels_val_path]:
    path.mkdir(parents=True, exist_ok=True)

# Get list of label files (exclude checkpoint and unrelated files)
label_files = [f for f in labels_raw_path.glob("*.txt") if f.name != "classes.txt"]

# Shuffle and split 80/20
random.shuffle(label_files)
split_index = int(0.8 * len(label_files))
train_files = label_files[:split_index]
val_files = label_files[split_index:]

# Define helper function to copy pairs
def copy_image_label_pair(label_file, split="train"):
    img_name = label_file.stem + ".JPG"
    img_src = raw_images_path / img_name
    if not img_src.exists():
        # Try lowercase .jpg
        img_src = raw_images_path / img_name.lower()
    if not img_src.exists():
        print(f"⚠️ Image missing for: {label_file.name}")
        return
    
    if split == "train":
        shutil.copy(label_file, labels_train_path / label_file.name)
        shutil.copy(img_src, images_train_path / img_src.name)
    else:
        shutil.copy(label_file, labels_val_path / label_file.name)
        shutil.copy(img_src, images_val_path / img_src.name)

# Copy files accordingly
for f in train_files:
    copy_image_label_pair(f, split="train")

for f in val_files:
    copy_image_label_pair(f, split="val")

# Output a summary
import pandas as pd
summary_df = pd.DataFrame({
    "Split": ["Train", "Validation"],
    "Image Count": [len(train_files), len(val_files)]
})
display(summary_df)


,Split,Image Count
0,Train,5
1,Validation,2
